# Project 4 手把拍照與道路目標標記

保留 Project 3 車道辨識，按手把 13 號鍵拍一張「未畫線原圖」，以該幀綠點為目標標籤；長按不連拍。

預設按 5 號鍵以速度 0.3 前進 1 秒，1 號鍵停止。按鍵名稱依手把而異，請看 Controller 的編號；這是手動定時移動，沒有自動循線或新增轉向。Run All 不會自動開車。

新資料在 `project4_data/dataset_xy`，辨識檢查圖在 `project4_data/review`，每張圖的紀錄在 `project4_data/metadata`。不要把紅線／綠點畫進訓練圖，也不要混用 Project 3 的舊 dataset_xy。

先開始辨識，再按一下手把任一鍵，手把會自動連線並綁定按鍵。確認綠點正確才按拍照；辨識失敗時暫停，點左側原圖的道路目標即可手動標記並儲存。累積目標 200 張，達標仍可繼續拍。

**標記規則（重要）**：目標點一律放在高度 65% 的那條參考線上（左圖的黃色虛線），只決定左右位置。自動標記本來就是這個高度；手動點選時預設會鎖定 y，點哪個高度都會被換成 65%，這樣整份資料的「往前看多遠」才一致。

車道線偵測程式在同資料夾的 `lane_label.py`（可以在 PC 上用 `python lane_label.py eval` 對既有資料集測試），這本 notebook 要和它放在同一個資料夾。

## 1. 載入套件
保留原檔提供的初始化程式。Notebook 使用 Python 3.6 可解析的語法。


In [ ]:
"""Provided setup cell. Do not modify."""
# Notebook display and widget utilities.
import ipywidgets
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display

# JetBot camera and robot interfaces.
from jetbot import Robot, Camera, bgr8_to_jpeg

# General utilities and image-processing packages.
from uuid import uuid1
import os
import json
import glob
import datetime
import numpy as np
import cv2
import time

import hashlib


## 2. 初始化相機與資料集
保留原檔提供的設定，既有照片不刪除。相機初始化每個 Kernel 執行一次；停止相機後若要重啟，請重新啟動 Kernel。


In [ ]:
"""Provided setup cell. Do not modify."""
from jupyter_clickable_image_widget import ClickableImageWidget

DATASET_DIR = os.path.join("project4_data", "dataset_xy")

# Create the dataset directory; keep existing files if it already exists.
try:
    os.makedirs(DATASET_DIR)
except FileExistsError:
    print("Directories not created because they already exist")

camera = Camera()


## 3. 室內膠帶参数

`LANE_PROFILE` 預設為 `INDOOR_PROFILE`（定義在 `lane_label.py`，參數與原本相同）。工作寬度 480 px，ROI 從高度 12% 開始，涵蓋下方兩側膠帶；不要求膠帶為低飽和白色，也不要求朝向固定消失點。要調整參數可以在這裡改 `LANE_PROFILE['contrast'] = …`。

- `contrast=8`：膠帶相對鄰近地墊的亮度差。
- `min_gray=60`：允許偏暗／偏紫的膠帶。
- `component_min_height=0.10`：排除地墊紋理等零碎亮點。
- `fit_tolerance=8`：曲線擬合容許離群列的像素差。
- `max_draw_gap=0.025`：長斷口不補線。

`CENTERLINE_CONFIG`：近端 (50%, 95%)，遠端觀察高度 `far_y=0.65`。

資料品質相關：

- `MANUAL_LOCK_Y=True`：手動點選時 y 固定在 `far_y`，只取點的 x。介面上也有勾選框可以臨時關掉。
- `NEAR_DUP_THRESHOLD=3.0`：和上一張存的圖幾乎一樣（32×32 灰階平均差小於這個值）時先擋下來，再按一次才存。設 0 關閉。
- `X_BINS=8`：介面上「標籤 x 分布」的格數。

手把：

- `GAMEPAD_INDEX=None`：自動偵測 0~3 號手把，哪個先連上就用哪個。固定用某一支就填它的編號。
- `CAPTURE_BUTTON` / `MOVE_BUTTON` / `STOP_BUTTON`：拍照、前進、停止的按鍵編號。手把連上後會自動綁定；斷線再連回來也會自動重綁。

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('.'))   # lane_label.py 和這本 notebook 放在同一個資料夾
from lane_label import (OUTDOOR_PROFILE_3, INDOOR_PROFILE, CENTERLINE_CONFIG as DEFAULT_CENTERLINE,
                        detect, compute_midpoint, draw_midpoint, draw_guides)

LANE_PROFILE = dict(INDOOR_PROFILE)       # 要恢復戶外設定可改為 dict(OUTDOOR_PROFILE_3)
MAX_FPS = 5.0
SHOW_MIDPOINT = True

# 座標為畫面寬／高比例：y 越大越靠近車子。
CENTERLINE_CONFIG = dict(DEFAULT_CENTERLINE)   # near (0.5, 0.95)、far_y 0.65、min_lane_width 0.10

# 資料品質
MANUAL_LOCK_Y = True          # 手動點選只取 x，y 固定在 far_y
NEAR_DUP_THRESHOLD = 3.0      # 和上一張幾乎一樣時要再按一次才存；0 = 關閉
X_BINS = 8                    # 標籤 x 分布的格數

# 課堂範例使用 13 號鍵拍照；不同手把請依 Controller 顯示調整。
GAMEPAD_INDEX = None          # None = 自動偵測 0~3 號；固定某一支就填編號
CAPTURE_BUTTON = 13
MOVE_BUTTON = 5
STOP_BUTTON = 1
MOVE_SPEED = 0.3
MOVE_SECONDS = 1.0
TARGET_COUNT = 200
MAX_CAPTURE_AGE = 1.5
CAPTURE_DEBOUNCE = 0.4

## 4 車道辨識與標籤分布

車道辨識（Canny、HoughLinesP 與曲線整理）在 `lane_label.py`，演算法和原本相同。藍點固定在高度 95%，綠點為高度 65% 的左右中點。這是標籤建議，需由拍攝者判斷是否為合理行進目標。

用既有 200 張測試的結果：綠點只在約 37% 的畫面算得出來，主要失敗在彎道。彎道時前方膠帶幾乎是橫的，內側膠帶在畫面外，65% 高度上根本沒有兩條線。所以**彎道、髮夾彎、偏離後回正的畫面要暫停後手動點選**，y 會自動鎖在 65%。

下面的函式統計目前資料集的標籤 x 分布，介面上會顯示長條圖；紅色的格子表示資料少於平均一半，優先補拍那個方向。

In [ ]:
def label_distribution(dataset_dir, bins=X_BINS, width=None):
    """每一格（畫面由左到右均分）有幾張標籤。"""
    width = width or camera.width
    counts = [0] * bins
    for path in glob.glob(os.path.join(dataset_dir, 'xy_*.jpg')):
        try:
            x = int(os.path.basename(path).split('_')[1])
        except (IndexError, ValueError):
            continue
        counts[min(bins - 1, max(0, x * bins // width))] += 1
    return counts


def render_distribution(dataset_dir, highlight_x=None):
    """標籤 x 分布的 HTML 長條圖；highlight_x 是剛存的那張，用藍框標出所在格子。"""
    counts = label_distribution(dataset_dir)
    bins = len(counts)
    biggest = max(counts) or 1
    average = sum(counts) / float(bins)
    hit = None if highlight_x is None else min(bins - 1, max(0, int(highlight_x) * bins // camera.width))
    bars = []
    for i, c in enumerate(counts):
        color = '#d9534f' if c < 0.5 * average else '#5cb85c'
        border = 'outline:2px solid #337ab7;' if i == hit else ''
        bars.append('<div style="display:inline-block;width:30px;margin:0 2px;text-align:center;vertical-align:bottom">'
                    '<div style="font-size:11px">%d</div>'
                    '<div style="height:%dpx;background:%s;%s"></div></div>'
                    % (c, max(1, int(60 * c / biggest)), color, border))
    return ('<div><b>標籤 x 分布</b>（畫面左 → 右，共 %d 張；紅色＝少於平均一半，優先補拍）<br>%s'
            '<div style="font-size:11px;color:#777">左側格子多＝目標常在左邊（左彎或車子偏右）；右側同理。</div></div>'
            % (sum(counts), ''.join(bars)))

## 5 介面與按鍵
手把會自動連線、自動綁定：**按一下手把任一鍵**（瀏覽器規定按過鍵網頁才讀得到手把），看到「已綁定」就可以用。斷線再連回來也會自動重綁，綁定時一律先取消「允許移動」。「重新綁定手把」按鈕只在改了按鍵編號、或自動綁定沒反應時使用。

Jupyter 分頁要停在前景：瀏覽器在背景分頁會暫停讀取手把。

拍照不要求勾選允許移動。前進需同時勾選允許移動且辨識正在執行。

In [ ]:
previous = globals().get('lane_controller')
if previous is not None:
    previous.close()

camera_widget = ClickableImageWidget(width=camera.width, height=camera.height)
img_widget = ipywidgets.Image(format='jpeg', width=camera.width, height=camera.height)
snapshot_widget = ipywidgets.Image(format='jpeg', width=camera.width, height=camera.height)
count_widget = ipywidgets.IntText(description='已存張數', disabled=True)
count_widget.value = len(glob.glob(os.path.join(DATASET_DIR, '*.jpg')))
status_widget = ipywidgets.Label(value='尚未開始，請按開始辨識。')
save_status_widget = ipywidgets.Label(value='可暫停後點左圖，或按儲存目前結果。')
start_button = ipywidgets.Button(description='開始辨識', button_style='success')
pause_button = ipywidgets.Button(description='暫停辨識', button_style='warning')
save_button = ipywidgets.Button(description='拍照存訓練原圖', button_style='info')
stop_camera_button = ipywidgets.Button(description='停止相機', button_style='danger')

# 每個編號一個 Controller；前端建立後就會在背景等待手把，不必顯示出來。
gamepad_indices = list(range(4)) if GAMEPAD_INDEX is None else [GAMEPAD_INDEX]
controllers = [ipywidgets.Controller(index=i) for i in gamepad_indices]
controller_box = ipywidgets.Box([controllers[0]])   # 顯示目前使用中的手把
bind_gamepad_button = ipywidgets.Button(description='重新綁定手把')
gamepad_status = ipywidgets.Label(value='等待手把：請按手把任一鍵（按過鍵網頁才讀得到手把）。')
move_enabled = ipywidgets.Checkbox(value=False, description='允許移動')
move_button = ipywidgets.Button(description='前進一次')
stop_move_button = ipywidgets.Button(description='停止移動', button_style='danger')
motion_status = ipywidgets.Label(value='未啟用移動')
progress_widget = ipywidgets.IntProgress(value=0, min=0, max=TARGET_COUNT, description='收集進度')
progress_text = ipywidgets.Label(value='0 / {} 張'.format(TARGET_COUNT))
lock_y_widget = ipywidgets.Checkbox(value=MANUAL_LOCK_Y, description='手動點選鎖定 y（只決定左右）',
                                    layout=ipywidgets.Layout(width='320px'), style={'description_width': 'initial'})
dist_widget = ipywidgets.HTML()


## 6 同幀資料收集
按鍵儲存最近已發布影格的原始 JPEG，檔名為 `xy_<x>_<y>_<uuid>.jpg`。座標單位為原始圖片像素，可供課堂 XYDataset 讀取。

原圖、辨識結果、中點均源自同一幀。運行時拒絕超過 1.5 秒的影格，重複同幀不再存；暫停後可檢查畫面再拍。手動標記只允許暫停時點左圖，點一下即存一張。

- 左圖（點選用）會畫出 65% 高度的黃色虛線，**只畫在顯示用的圖上**，存進訓練集的原圖沒有任何疊圖。
- 手動點選的座標用圖片在瀏覽器上實際顯示的大小換算，視窗太窄、圖片被縮小時也不會偏。
- 和上一張存的畫面幾乎一樣時會先擋下來並提示，要存就再按一次。
- 剛開相機還沒有影格時會等待（最多 10 秒），不會直接報錯停止。

這不是相機硬體快門，存的是按下時最近完成辨識的畫面，可能有運算／瀏覽器延遲；需要精準確認時停車再暫停辨識。確認右側檢查圖的黃色圈位置正確。

In [ ]:
import asyncio
import copy
from concurrent.futures import ThreadPoolExecutor


def process_camera_frame(frame, profile, show_midpoint):
    """背景執行緒只計算影像；不在背景執行緒修改 widget。

    回傳 (原圖 JPEG, 辨識結果 JPEG, metadata, 點選用顯示圖 JPEG)。
    原圖 JPEG 就是存進訓練集的那張，不畫任何東西；參考線只畫在顯示圖上。
    """
    height, width = frame.shape[:2]
    line_img, _, metadata = detect(frame, profile)
    state = compute_midpoint(metadata, width, height, CENTERLINE_CONFIG)
    midpoint = draw_midpoint(line_img, state) if show_midpoint else None
    metadata['midpoint'] = list(midpoint) if midpoint is not None else None
    # 近似重複判斷用的小縮圖（不寫進 JSON）
    metadata['_thumb'] = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (32, 32),
                                    interpolation=cv2.INTER_AREA).astype(np.float32)
    guide = draw_guides(frame, CENTERLINE_CONFIG['far_y'], lock_y=lock_y_widget.value)
    return (bytes(bgr8_to_jpeg(frame)), bytes(bgr8_to_jpeg(line_img)), metadata,
            bytes(bgr8_to_jpeg(guide)))


class LiveLaneController:
    """每次最多處理一幀，不排隊累積過期影格；畫面更新回到 Notebook 事件迴圈。"""

    def __init__(self, camera, raw_widget, result_widget, saved_widget,
                 count_widget, status_widget, save_status_widget, dataset_dir,
                 profile, max_fps=5.0, show_midpoint=True, processor=None, loop=None):
        if max_fps <= 0:
            raise ValueError('max_fps 必須大於 0')
        self.camera = camera
        self.raw_widget = raw_widget
        self.result_widget = result_widget
        self.saved_widget = saved_widget
        self.count_widget = count_widget
        self.status_widget = status_widget
        self.save_status_widget = save_status_widget
        self.dataset_dir = dataset_dir
        self.profile = copy.deepcopy(profile)
        self.max_fps = float(max_fps)
        self.show_midpoint = bool(show_midpoint)
        self.processor = processor or process_camera_frame
        self.loop = loop or asyncio.get_event_loop()
        self.executor = ThreadPoolExecutor(max_workers=1)
        self.task = None
        self.running = False
        self.closed = False
        self.latest = None
        self.frame_id = 0
        self.epoch = 0
        self.bindings = []
        os.makedirs(dataset_dir, exist_ok=True)

    def start(self, button=None):
        if self.closed:
            return
        if not self.running:
            self.epoch += 1
        self.running = True
        self.status_widget.value = '辨識中：等待最新影格…'
        if self.task is None or self.task.done():
            self.task = self.loop.create_task(self._run())

    def pause(self, button=None):
        self.running = False
        self.epoch += 1
        self.status_widget.value = '已暫停，保留最後一幀；可存圖或按開始繼續。'

    async def _run(self):
        try:
            while self.running and not self.closed:
                started = self.loop.time()
                epoch = self.epoch
                frame = self.camera.value
                if (not isinstance(frame, np.ndarray) or frame.ndim != 3
                        or frame.shape[2] != 3 or frame.size == 0):
                    # 剛開相機時可能還沒有影格：等一下再試，超過 10 秒才放棄。
                    waited = getattr(self, '_waited', 0.0)
                    if waited > 10.0:
                        raise ValueError('相機超過 10 秒沒有提供有效的 BGR 影格')
                    self._waited = waited + 0.2
                    self.status_widget.value = '等待相機影格…'
                    await asyncio.sleep(0.2)
                    continue
                self._waited = 0.0
                frame = frame.copy()
                captured_at = datetime.datetime.now().isoformat()
                captured_monotonic = time.monotonic()
                raw_bytes, result_bytes, metadata, display_bytes = await self.loop.run_in_executor(
                    self.executor, self.processor, frame, self.profile, self.show_midpoint)
                # 暫停、重建介面或快速重新開始後，不發布舊的運算結果。
                if self.closed or not self.running:
                    break
                if epoch != self.epoch:
                    continue
                self.frame_id += 1
                self.latest = {'raw_jpeg': raw_bytes, 'result_jpeg': result_bytes,
                               'metadata': metadata, 'frame_id': self.frame_id,
                               'captured_at': captured_at, 'captured_monotonic': captured_monotonic, 'size': [frame.shape[1], frame.shape[0]]}
                self.raw_widget.value = display_bytes   # 點選用：有參考線；存檔用的是 raw_bytes
                self.result_widget.value = result_bytes
                elapsed = max(self.loop.time() - started, 0.001)
                midpoint = metadata.get('midpoint')
                self.status_widget.value = (
                    '{} | 第 {} 幀 | {} 組標線 | 單幀 {:.0f} ms | 中點 {}'
                    .format(self.profile.get('name', 'profile_3'), self.frame_id, metadata['lane_group_count'], elapsed * 1000,
                            midpoint if midpoint is not None else '無足夠左右標線'))
                await asyncio.sleep(max(0.0, 1.0 / self.max_fps - elapsed))
        except asyncio.CancelledError:
            pass
        except Exception as exc:
            self.running = False
            if not self.closed:
                self.status_widget.value = '辨識暫停：{}: {}'.format(type(exc).__name__, exc)

    def save(self, click=None):
        """儲存最近發布的同一幀結果；暂停後再點選可固定標註影格。"""
        record = self.latest
        if record is None:
            self.save_status_widget.value = '尚無辨識結果，請先按開始辨識。'
            return
        try:
            click_xy = None
            if click is not None:
                width, height = record['size']
                x = float(click['offsetX'])
                y = float(click['offsetY'])
                if not np.isfinite(x) or not np.isfinite(y):
                    raise ValueError('點選座標無效')
                x = int(round(x * width / float(self.raw_widget.width)))
                y = int(round(y * height / float(self.raw_widget.height)))
                click_xy = [max(0, min(width - 1, x)), max(0, min(height - 1, y))]
            stem = ('xy_%03d_%03d_' % tuple(click_xy) if click_xy is not None else 'lane_') + str(uuid1())
            path = os.path.join(self.dataset_dir, stem + '.jpg')
            with open(path, 'wb') as f:
                f.write(record['result_jpeg'])
            sidecar = {'frame_id': record['frame_id'], 'captured_at': record['captured_at'],
                       'click_xy': click_xy, 'input_size': record['size'], 'profile_id': self.profile.get('name', 'profile_3'),
                       'centerline': record['metadata'].get('centerline'),
                       'midpoint': record['metadata'].get('midpoint'),
                       'lane_group_count': record['metadata']['lane_group_count'],
                       'profile': self.profile}
            with open(os.path.join(self.dataset_dir, stem + '.json'), 'w', encoding='utf-8') as f:
                json.dump(sidecar, f, ensure_ascii=False, indent=2)
            self.saved_widget.value = record['result_jpeg']
            self.count_widget.value = len(glob.glob(os.path.join(self.dataset_dir, '*.jpg')))
            self.save_status_widget.value = '已儲存第 {} 幀：{}'.format(record['frame_id'], stem + '.jpg')
        except Exception as exc:
            self.save_status_widget.value = '存圖失敗：{}: {}'.format(type(exc).__name__, exc)

    def save_button(self, button=None):
        self.save()

    def save_click(self, widget, content, buffers):
        if content.get('event') == 'click':
            self.save(content.get('eventData', {}))

    def shutdown_camera(self, button=None):
        self.close()
        try:
            self.camera.stop()
            self.status_widget.value = '相機已停止。再次使用請重新啟動 Kernel 並依序執行。'
        except Exception as exc:
            self.status_widget.value = '辨識已停止，但相機停止失敗：{}'.format(exc)

    def bind(self, start_button, pause_button, save_button):
        for button, callback in ((start_button, self.start), (pause_button, self.pause),
                                 (save_button, self.save_button)):
            button.on_click(callback)
            self.bindings.append((button, 'on_click', callback))
        self.raw_widget.on_msg(self.save_click)
        self.bindings.append((self.raw_widget, 'on_msg', self.save_click))

    def close(self):
        if self.closed:
            return
        self.closed = True
        self.running = False
        self.epoch += 1
        for widget, method, callback in self.bindings:
            getattr(widget, method)(callback, remove=True)
        self.bindings = []
        if self.task is not None and not self.task.done():
            self.task.cancel()
        self.executor.shutdown(wait=False)




class DatasetController(LiveLaneController):
    """保存未畫線原圖；標籤與影像必須來自同一個已發布影格。"""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.session_id = str(uuid1())
        self.saved_frames = set()
        self.last_save_time = -float('inf')
        self.drive = None
        self.gamepad_binding = None
        self.aux_bindings = []
        self.last_thumb = self._newest_thumb()
        self.dup_pending = None
        self.review_dir = os.path.join(os.path.dirname(self.dataset_dir), 'review')
        self.meta_dir = os.path.join(os.path.dirname(self.dataset_dir), 'metadata')
        for directory in (self.review_dir, self.meta_dir):
            os.makedirs(directory, exist_ok=True)
        self.update_count()

    def _newest_thumb(self):
        """上一次存的圖（重開 kernel 後也能判斷近似重複）。"""
        paths = glob.glob(os.path.join(self.dataset_dir, 'xy_*.jpg'))
        if not paths:
            return None
        image = cv2.imread(max(paths, key=os.path.getmtime), cv2.IMREAD_GRAYSCALE)
        if image is None:
            return None
        return cv2.resize(image, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)

    def update_count(self, highlight_x=None):
        dist_widget.value = render_distribution(self.dataset_dir, highlight_x)
        count = len(glob.glob(os.path.join(self.dataset_dir, 'xy_*.jpg')))
        self.count_widget.value = count
        progress_widget.max = max(TARGET_COUNT, count)
        progress_widget.value = count
        progress_text.value = '{} / {} 張{}'.format(count, TARGET_COUNT,
            '（已達目標，仍可繼續）' if count >= TARGET_COUNT else '')

    def save(self, click=None):
        if self.closed:
            return
        record = self.latest
        if record is None:
            self.save_status_widget.value = '尚無影格，請先開始辨識。'
            return
        now = time.monotonic()
        if self.running and now - record['captured_monotonic'] > MAX_CAPTURE_AGE:
            self.save_status_widget.value = '影格過舊，請等辨識畫面更新後再拍。'
            return
        if record['frame_id'] in self.saved_frames:
            self.save_status_widget.value = '這一幀已存過；請開始辨識取得新畫面。'
            return
        if now - self.last_save_time < CAPTURE_DEBOUNCE:
            return
        try:
            width, height = record['size']
            if click is not None:
                if self.running:
                    raise ValueError('手動標記請先暫停辨識，再點左側原圖的道路目標位置')
                # 用圖片在瀏覽器上實際顯示的大小換算（視窗窄時圖片會被縮小）；舊版 widget 沒有就用設定值。
                shown_w = float(click.get('width') or self.raw_widget.width)
                shown_h = float(click.get('height') or self.raw_widget.height)
                x = float(click['offsetX']) * width / shown_w
                y = float(click['offsetY']) * height / shown_h
                if not np.isfinite([x, y]).all():
                    raise ValueError('座標無效')
                target = [int(np.clip(round(x), 0, width - 1)),
                          int(np.clip(round(y), 0, height - 1))]
                source = 'manual_click'
                if lock_y_widget.value:
                    target[1] = int(round((height - 1) * CENTERLINE_CONFIG['far_y']))
                    source = 'manual_click_y_locked'
            else:
                centerline = record['metadata'].get('centerline', {})
                target = centerline.get('far_point')
                if not centerline.get('valid') or target is None:
                    raise ValueError('沒有有效中點，未儲存；可暫停後點左圖手動標記')
                target = [int(v) for v in target]
                source = 'hough_midpoint_confirmed_by_button'
            if not (0 <= target[0] < width and 0 <= target[1] < height):
                raise ValueError('目標超出圖片範圍')
            thumb = record['metadata'].get('_thumb')
            if NEAR_DUP_THRESHOLD > 0 and thumb is not None and self.last_thumb is not None:
                diff = float(np.mean(np.abs(thumb - self.last_thumb)))
                if diff < NEAR_DUP_THRESHOLD and self.dup_pending != record['frame_id']:
                    self.dup_pending = record['frame_id']
                    raise ValueError('畫面和上一張幾乎一樣（差異 {:.1f}）。移動或轉一下車子再拍；'
                                     '確定要存就再按一次。'.format(diff))
            self.dup_pending = None
            stem = 'xy_%03d_%03d_%s' % (target[0], target[1], uuid1())
            image_path = os.path.join(self.dataset_dir, stem + '.jpg')
            review_path = os.path.join(self.review_dir, stem + '.jpg')
            meta_path = os.path.join(self.meta_dir, stem + '.json')
            # 檢查圖不放在訓練資料夾；人工標記另以黃色圈顯示實際標籤。
            review = cv2.imdecode(np.frombuffer(record['result_jpeg'], np.uint8), 1)
            cv2.circle(review, tuple(target), 6, (0, 255, 255), 2)
            review_bytes = bytes(bgr8_to_jpeg(review))
            info = {'session_id': self.session_id, 'frame_id': record['frame_id'],
                    'captured_at': record['captured_at'], 'saved_at': datetime.datetime.now().isoformat(),
                    'target_xy': target, 'label_source': source, 'input_size': record['size'],
                    'y_locked': source == 'manual_click_y_locked',
                    'centerline': record['metadata'].get('centerline'),
                    'profile': self.profile, 'centerline_config': dict(CENTERLINE_CONFIG),
                    'image_sha256': hashlib.sha256(record['raw_jpeg']).hexdigest()}
            created = []
            try:
                for path, content in ((meta_path, json.dumps(info, ensure_ascii=False, indent=2).encode('utf-8')),
                                      (review_path, review_bytes), (image_path, record['raw_jpeg'])):
                    with open(path, 'xb') as f:
                        created.append(path)
                        f.write(content)
            except Exception:
                for path in created:
                    os.remove(path)
                raise
            self.saved_frames.add(record['frame_id'])
            self.last_save_time = now
            if thumb is not None:
                self.last_thumb = thumb
            self.saved_widget.value = review_bytes
            self.update_count(highlight_x=target[0])
            self.save_status_widget.value = '已存原圖：{}；標籤 {}（{}）'.format(stem + '.jpg', target, source)
        except Exception as exc:
            self.save_status_widget.value = '未儲存：{}'.format(exc)

    def pause(self, button=None):
        if self.drive is not None:
            self.drive.stop()
        super().pause(button)

    async def _run(self):
        try:
            await super()._run()
        finally:
            if self.drive is not None:
                self.drive.stop()

    def close(self):
        if self.closed:
            return
        if self.gamepad_binding is not None:
            self.gamepad_binding.close()
        if self.drive is not None:
            self.drive.stop()
        for widget, method, callback in self.aux_bindings:
            getattr(widget, method)(callback, remove=True)
        self.aux_bindings = []
        super().close()


class PulseDrive:
    """保留原本前進 1 秒的動作，改成按鍵觸發，不自動循線。"""
    def __init__(self, lane, enabled, robot_factory):
        self.lane = lane
        self.enabled = enabled
        self.robot_factory = robot_factory
        self.robot = None
        self.task = None

    def start(self, button=None):
        if self.lane.closed or not self.enabled.value or not self.lane.running:
            motion_status.value = '未移動：請先開始辨識，再勾選允許移動。'
            return
        record = self.lane.latest
        if record is None or time.monotonic() - record['captured_monotonic'] > MAX_CAPTURE_AGE:
            motion_status.value = '未移動：等待新影格。'
            return
        if self.task is not None and not self.task.done():
            return
        self.task = self.lane.loop.create_task(self._pulse())

    async def _pulse(self):
        try:
            if self.robot is None:
                self.robot = self.robot_factory()
            self.robot.forward(MOVE_SPEED)
            motion_status.value = '前進中，可按拍照；到時自動停止。'
            await asyncio.sleep(MOVE_SECONDS)
        except asyncio.CancelledError:
            pass
        except Exception as exc:
            motion_status.value = '移動錯誤：{}'.format(exc)
        finally:
            self.stop_motor()

    def stop_motor(self):
        if self.robot is not None:
            try:
                self.robot.stop()
                motion_status.value = '車子已停止。'
            except Exception as exc:
                motion_status.value = '停止指令失敗：{}'.format(exc)

    def stop(self, button=None):
        if self.task is not None and not self.task.done():
            self.task.cancel()
        self.stop_motor()


class GamepadBinding:
    """手把自動連線與自動綁定。

    - 同時監看多個 Controller（GAMEPAD_INDEX=None 時是 0~3 號），第一個連上且按鍵數量足夠的就用它。
    - 連上 → 自動綁定拍照／前進／停止；斷線 → 解除綁定並停車；再連上 → 自動重綁。
    - 每次綁定都先取消「允許移動」，避免一連上就開走。
    """
    def __init__(self, lane, controllers):
        self.lane = lane
        self.controllers = list(controllers)
        self.indices = (CAPTURE_BUTTON, MOVE_BUTTON, STOP_BUTTON)
        self.active = None
        self.bindings = []
        self.observers = []
        for c in self.controllers:
            def observer(change, c=c):
                self._changed(c, change)
            c.observe(observer, names=['connected', 'buttons'])
            self.observers.append((c, observer))
        move_enabled.observe(self.enable_changed, names='value')
        if len(set(self.indices)) != 3 or min(self.indices) < 0:
            gamepad_status.value = '拍照、移動、停止按鍵編號需不同且不小於零，請修改第 3 節後重新執行。'
            return
        # 重新執行這格時手把可能早就連著了
        self._activate_first_ready()

    # ---- 狀態 -------------------------------------------------------------
    def _ready(self, c):
        return bool(c.connected) and len(c.buttons) > max(self.indices)

    def _name(self, c):
        return '{} 號手把{}'.format(c.index, '（{}）'.format(c.name) if c.name else '')

    def _changed(self, c, change):
        if c is self.active:
            if not c.connected:
                self._lost('{}已斷線，已停止移動；重新連上後按任一鍵會自動綁定。'.format(self._name(c)))
            elif change['name'] == 'buttons':
                # 按鍵列表換了（例如重新連線）：先停車再自動重綁
                if self._ready(c):
                    self._bind(c)
                else:
                    self._lost('{}的按鍵列表不足，已停止移動。'.format(self._name(c)))
            return
        if self.active is None:
            if self._ready(c):
                self._activate(c)
            elif c.connected and c.buttons:
                gamepad_status.value = '{}只有 {} 個按鍵，不夠綁定 {}／{}／{} 號；請改第 3 節的按鍵編號。'.format(
                    self._name(c), len(c.buttons), *self.indices)

    def _activate_first_ready(self):
        for c in self.controllers:
            if self._ready(c):
                self._activate(c)
                return True
        return False

    def _activate(self, c):
        self.active = c
        controller_box.children = [c]
        self._bind(c)

    def _lost(self, message):
        self.unbind()
        self.active = None
        self.stop_all()
        gamepad_status.value = message
        self._activate_first_ready()   # 有另一支已經連著就改用它

    # ---- 綁定 -------------------------------------------------------------
    def unbind(self):
        for button, callback in self.bindings:
            button.unobserve(callback, names='value')
        self.bindings = []

    def _bind(self, c):
        self.unbind()
        self.stop_all()
        actions = (self.lane.save_button, self.lane.drive.start, self.stop_all)
        for index, action in zip(self.indices, actions):
            # 只處理由放開到按下的邊緣，長按不連拍、不重複移動。
            def callback(change, action=action):
                if float(change.get('old', 0)) <= 0.5 < float(change['new']):
                    action()
            target = c.buttons[index]
            target.observe(callback, names='value')
            self.bindings.append((target, callback))
        gamepad_status.value = '已自動綁定{}：{} 拍照／{} 前進／{} 停止。'.format(self._name(c), *self.indices)

    def bind(self, button=None):
        """「重新綁定手把」按鈕：重新讀取按鍵編號設定並綁定。"""
        self.indices = (CAPTURE_BUTTON, MOVE_BUTTON, STOP_BUTTON)
        if len(set(self.indices)) != 3 or min(self.indices) < 0:
            gamepad_status.value = '拍照、移動、停止按鍵編號需不同且不小於零。'
            return
        if self.active is not None and self._ready(self.active):
            self._bind(self.active)
        elif not self._activate_first_ready():
            gamepad_status.value = '還沒讀到手把：請按手把任一鍵；Jupyter 分頁要停在前景。'

    def enable_changed(self, change):
        if not change['new']:
            self.lane.drive.stop()

    def stop_all(self, button=None):
        move_enabled.value = False
        self.lane.drive.stop()

    def close(self):
        self.unbind()
        for c, observer in self.observers:
            c.unobserve(observer, names=['connected', 'buttons'])
        self.observers = []
        move_enabled.unobserve(self.enable_changed, names='value')


# 重新執行時，解除舊按鍵與背景工作。
previous = globals().get('lane_controller')
if previous is not None:
    previous.close()
lane_controller = DatasetController(
    camera, camera_widget, img_widget, snapshot_widget,
    count_widget, status_widget, save_status_widget, DATASET_DIR,
    LANE_PROFILE, max_fps=MAX_FPS, show_midpoint=True)
lane_controller.drive = PulseDrive(lane_controller, move_enabled, Robot)
lane_controller.gamepad_binding = GamepadBinding(lane_controller, controllers)
lane_controller.bind(start_button, pause_button, save_button)
for button, callback in ((stop_camera_button, lane_controller.shutdown_camera),
                         (bind_gamepad_button, lane_controller.gamepad_binding.bind),
                         (move_button, lane_controller.drive.start),
                         (stop_move_button, lane_controller.gamepad_binding.stop_all)):
    button.on_click(callback)
    lane_controller.aux_bindings.append((button, 'on_click', callback))
print('資料集目錄：' + os.path.abspath(DATASET_DIR))
print('預設不移動。先開始辨識，再按一下手把任一鍵，會自動連線並綁定。')


## 7 顯示與操作
依序執行後，開始辨識 → 手把按任一鍵（自動連線、自動綁定）→ 確認綠點 → 13 號鍵拍照。
需要移動時勾選允許移動，再按 5 號鍵；1 號鍵停止並取消允許移動。暫停辨識、停止相機或手把斷線也會發出停止指令。

拍攝建議：

1. 直線段：綠點合理時直接按 13 號鍵（自動標記）。
2. 彎道、髮夾彎、偏離道路中心、斜著快壓線的畫面：**暫停辨識 → 點左圖上道路該去的方向**。y 會鎖在黃色虛線上，只要決定左右；點完右側檢查圖會出現黃色圈。
3. 偏離與回正的畫面要自己把車擺過去（5 號鍵只會直線前進）。偏左、偏右、車頭朝向膠帶都要拍。
4. 每存一張，下方的「標籤 x 分布」會更新；**紅色的格子優先補拍**。

In [ ]:
data_collection_widget = ipywidgets.VBox([
    ipywidgets.HBox([start_button, pause_button, save_button, stop_camera_button]),
    ipywidgets.HBox([
        ipywidgets.VBox([ipywidgets.Label(value='目前處理的原圖'), camera_widget]),
        ipywidgets.VBox([ipywidgets.Label(value='動態辨識結果'), img_widget]),
        ipywidgets.VBox([ipywidgets.Label(value='上次標記檢查圖（不供訓練）'), snapshot_widget]),
    ]),
    controller_box,
    ipywidgets.HBox([bind_gamepad_button, move_enabled, move_button, stop_move_button]),
    lock_y_widget,
    gamepad_status,
    motion_status,
    progress_widget,
    progress_text,
    dist_widget,
    count_widget,
    status_widget,
    save_status_widget,
])
display(data_collection_widget)


## 8 收集完成與訓練銜接
- 訓練照片：`project4_data/dataset_xy/xy_*.jpg`；檢查圖與 JSON 分別位於同層 review、metadata，不送入訓練。
- **搬到 PC 時，`dataset_xy` 和 `metadata` 兩個資料夾一起複製**，放在同一層，例如 `data/dataset_xy_<日期>/dataset_xy` 和 `data/dataset_xy_<日期>/metadata`。`train.py --split session` 會讀 `metadata` 裡的 `session_id` 和拍攝時間，照拍攝段落切 train / val，避免相鄰的相似照片分別落進兩邊。
- 訓練用 `train.py`（見 README）；`--data` 指向 `dataset_xy`。
- 200 張是起點，請包含直路、左右彎、偏左與偏右後需要回復的畫面。不要只在同一位置連拍。每拍完一段檢查 review，錯誤標籤的原圖不要拿去訓練（或寫進 `--exclude` 的清單）。
- 結束按停止相機；此檔不在 Run All 中執行一次性的前進命令。手把連線、按鍵編號及實機移動需在 JetBot 確認，本地測試不能替代硬體測試。